### Import dependencies

In [1]:
import openai
import os
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import SystemMessage, AIMessage
from langsmith import Client
from pydantic import BaseModel, Field
from typing import List, Annotated, Any
from operator import add
from langsmith import traceable
from jinja2 import Template


In [2]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

### Download example eval dataset from Langsmith



In [3]:
ls_client = Client()

In [4]:
dataset = ls_client.read_dataset(dataset_name="coordinator-dataset-evaluation")

In [5]:
reference_input = [item.inputs['input'] for item in list(ls_client.list_examples(dataset=dataset.id, limit=50))]
reference_output = [item.outputs for item in list(ls_client.list_examples(dataset=dataset.id, limit=50))]

### Coordinator agent evaluation

In [6]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to")
    task: str = Field(description="The task to be performed by the agent")
    
class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence")

class FinalAgentResponse(BaseModel):
    answer: str = Field(description="The answer to the user's question")

class AgentProperties(BaseModel):
    final_answer: bool = False
    iteration: int = 0

class CoordinatorProperties(BaseModel):
    final_answer: bool = False
    iteration: int = 0
    plan: List[Delegation] = []
    next_agent: str = ""

class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorProperties = CoordinatorProperties()
    answer: str = ""

In [25]:
@traceable(
    name= "coordinator_agent",
    run_type= "llm",
    metadata= {
        "ls_provider": "openai",
        "ls_model": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    prompt_template = """
    You are a Coordinator Agent as part of a shopping assistant.

    ## Instructions

    - Your role is to create plans for solving user queries and delegate the tasks accordingly.
    - You will be given a conversation history, your task is to create a plan for solving the user's query.
    - After the plan is created, you should output the next agent to invoke and the task to be performed by that agent.
    - Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and revise the plan.
    - If there is a sequence of tasks to be performed by a single agent, you should combine them into a single task.
    - Do not route to any agent if the user's query needs clarification or is irrelevant. Do it yourself.

    ## Available Agents

    - product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
    - shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
    - warehouse_manager_agent: The user is asking to reserve items from the warehouses or about the availability of the items in the warehouses.

    ## Examples

    Question: "Do you have running shoes under $100?"
    Next agent: product_qna_agent

    Question: "Can you list the items in my cart?"
    Next agent: shopping_cart_agent

    Question: "Can you reserve my shopping cart?"
    Next agent: warehouse_manager_agent
    """

    template = Template(prompt_template)
    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort = "low",
        use_responses_api=True,
    )

    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages #this lets the model reference the previous messages (the history)
        ]
    )

    final_answer = False
    answer = ""
    plan = []
    next_agent = ""

    def sanitize_response(response):
        for tool_call in response.tool_calls:
            if tool_call.get('name') == "FinalAgentResponse":
                answer = tool_call.get('args').get('answer')
        
        return AIMessage(content=answer)    

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get('name') == 'Plan':
            plan = response.tool_calls[0].get('args').get('plan')
            next_agent = response.tool_calls[0].get('args').get('next_agent')
            response = None
        else:   
            for tool_call in response.tool_calls:
                if tool_call.get('name') == 'FinalAgentResponse':
                    final_answer = True
                    answer = tool_call.get('args').get('answer')
                    response = sanitize_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "final_answer": final_answer,
            "iteration": state.coordinator_agent.iteration + 1,
            "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [ ]:
#Generate the answer for the first example to reproduce the failure
answer = coordinator_agent(
    State(
        messages=reference_input[0]['messages'],
        coordinator_agent=CoordinatorProperties(
            iteration=0,
            plan=[],
            next_agent="",
            final_answer=False
        ),
        answer=""
    )
)

In [28]:
answer

{'messages': [],
 'coordinator_agent': {'final_answer': False,
  'iteration': 1,
  'plan': [{'agent': 'product_qna_agent',
    'task': 'Find outdoor-use speaker options that are available in green, and summarize the best matches with relevant outdoor features like waterproof rating, portability, battery life, and rating.'}],
  'next_agent': 'product_qna_agent'},
 'answer': ''}

In [7]:
def evaluate_coordinator_delegation(run, example):
    final_answer_match = run['coordinator_agent']['final_answer'] == example['coordinator_agent']['final_answer']
    next_agent_match = run['coordinator_agent']['next_agent'] == example['coordinator_agent']['next_agent']

    return final_answer_match and next_agent_match

In [30]:
evaluate_coordinator_delegation(answer, reference_output[0])

False

### Run against Langsmith

In [8]:
def evaluate_coordinator_delegation_for_ls(run, example):
    final_answer_match = run.outputs['coordinator_agent']['final_answer'] == example.outputs['coordinator_agent']['final_answer']
    next_agent_match = run.outputs['coordinator_agent']['next_agent'] == example.outputs['coordinator_agent']['next_agent']

    return final_answer_match and next_agent_match

In [33]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x['input']['messages'],
            coordinator_agent=CoordinatorProperties(
                iteration=0,
                plan=[],
                next_agent="",
                final_answer=False
            ),
            answer=""
        )
    ),
    data='coordinator-dataset-evaluation',
    evaluators=[evaluate_coordinator_delegation_for_ls],
    experiment_prefix='coordinator-delegation',
    num_repetitions=5 #each example will be run 5 times and the results will be averaged
)

View the evaluation results for experiment: 'coordinator-delegation-500e86c8' at:
https://smith.langchain.com/o/a215ca14-ac83-59e9-9b9e-fa7ba18e0446/datasets/268622a0-304b-495a-b347-361d21088c56/compare?selectedSessions=819ddb4e-cbca-4907-b471-4bf42b58b397




15it [00:37,  2.53s/it]


### Making Coordinator Agent more robust

#### Option 1: Adding explicit robust routing rules to the prompt

In [9]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to")
    task: str = Field(description="The task to be performed by the agent")
    
class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence")

class FinalAgentResponse(BaseModel):
    answer: str = Field(description="The answer to the user's question")

class AgentProperties(BaseModel):
    final_answer: bool = False
    iteration: int = 0

class CoordinatorProperties(BaseModel):
    final_answer: bool = False
    iteration: int = 0
    plan: List[Delegation] = []
    next_agent: str = ""

class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorProperties = CoordinatorProperties()
    answer: str = ""

In [9]:
@traceable(
    name= "coordinator_agent",
    run_type= "llm",
    metadata= {
        "ls_provider": "openai",
        "ls_model": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    prompt_template = """
    You are a Coordinator Agent as part of a shopping assistant.

    ## Instructions

    - Your role is to delegate work to worker agents in order to solve user queries.
    - You should output the next agent to invoke.
    - Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool or output the final answer via the `FinalAgentResponse` tool.
    - Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
    - Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
    - Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
    - Do not delegate work to the same agent twice in a row.

    ## Routing rules

    Match the user's latest message to one of these patterns:

    1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
        Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
        → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

    2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
        Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
        → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

    3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
        Examples: "reserve these", "hold these for me", "check warehouse stock for X".
        → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

    Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
    "I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

    ## Available Agents

    - product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
    - shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
    - warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
    """

    template = Template(prompt_template)
    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort = "low",
        use_responses_api=True,
    )

    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages #this lets the model reference the previous messages (the history)
        ]
    )

    final_answer = False
    answer = ""
    plan = []
    next_agent = ""

    def sanitize_response(response):
        for tool_call in response.tool_calls:
            if tool_call.get('name') == "FinalAgentResponse":
                answer = tool_call.get('args').get('answer')
        
        return AIMessage(content=answer)    

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get('name') == 'Plan':
            plan = response.tool_calls[0].get('args').get('plan')
            next_agent = response.tool_calls[0].get('args').get('next_agent')
            response = None
        else:   
            for tool_call in response.tool_calls:
                if tool_call.get('name') == 'FinalAgentResponse':
                    final_answer = True
                    answer = tool_call.get('args').get('answer')
                    response = sanitize_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "final_answer": final_answer,
            "iteration": state.coordinator_agent.iteration + 1,
            "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [10]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x['input']['messages'],
            coordinator_agent=CoordinatorProperties(
                iteration=0,
                plan=[],
                next_agent="",
                final_answer=False
            ),
            answer=""
        )
    ),
    data='coordinator-dataset-evaluation',
    evaluators=[evaluate_coordinator_delegation_for_ls],
    experiment_prefix='coordinator-delegation-v1',
    num_repetitions=5 #each example will be run 5 times and the results will be averaged
)

/home/pcrespo/Documents/estudos/github_repos/e2e_ai_engineering_bootcamp/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'coordinator-delegation-v1-b2a9dd2e' at:
https://smith.langchain.com/o/a215ca14-ac83-59e9-9b9e-fa7ba18e0446/datasets/268622a0-304b-495a-b347-361d21088c56/compare?selectedSessions=a3f3ef7e-63ef-49d8-a026-22e9b402215b




15it [00:46,  3.09s/it]


#### Option 2: Increase reasoning effort

In [10]:
@traceable(
    name= "coordinator_agent",
    run_type= "llm",
    metadata= {
        "ls_provider": "openai",
        "ls_model": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    prompt_template = """
    You are a Coordinator Agent as part of a shopping assistant.

    ## Instructions

    - Your role is to delegate work to worker agents in order to solve user queries.
    - You should output the next agent to invoke.
    - Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool or output the final answer via the `FinalAgentResponse` tool.
    - Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
    - Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
    - Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
    - Do not delegate work to the same agent twice in a row.

    ## Routing rules

    Match the user's latest message to one of these patterns:

    1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
        Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
        → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

    2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
        Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
        → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

    3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
        Examples: "reserve these", "hold these for me", "check warehouse stock for X".
        → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

    Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
    "I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

    ## Available Agents

    - product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
    - shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
    - warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
    """

    template = Template(prompt_template)
    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort = "medium",
        use_responses_api=True,
    )

    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages #this lets the model reference the previous messages (the history)
        ]
    )

    final_answer = False
    answer = ""
    plan = []
    next_agent = ""

    def sanitize_response(response):
        for tool_call in response.tool_calls:
            if tool_call.get('name') == "FinalAgentResponse":
                answer = tool_call.get('args').get('answer')
        
        return AIMessage(content=answer)    

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get('name') == 'Plan':
            plan = response.tool_calls[0].get('args').get('plan')
            next_agent = response.tool_calls[0].get('args').get('next_agent')
            response = None
        else:   
            for tool_call in response.tool_calls:
                if tool_call.get('name') == 'FinalAgentResponse':
                    final_answer = True
                    answer = tool_call.get('args').get('answer')
                    response = sanitize_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "final_answer": final_answer,
            "iteration": state.coordinator_agent.iteration + 1,
            "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

#### Option 3: Removing the "plan" from the Plan
- the plan might be a source of errors, as the coordinator will be forced to use an agent that might not be the best fit for the task

In [13]:
class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")

In [14]:
@traceable(
    name= "coordinator_agent",
    run_type= "llm",
    metadata= {
        "ls_provider": "openai",
        "ls_model": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    prompt_template = """
    You are a Coordinator Agent as part of a shopping assistant.

    ## Instructions

    - Your role is to delegate work to worker agents in order to solve user queries.
    - You should output the next agent to invoke.
    - Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool or output the final answer via the `FinalAgentResponse` tool.
    - Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
    - Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
    - Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
    - Do not delegate work to the same agent twice in a row.

    ## Routing rules

    Match the user's latest message to one of these patterns:

    1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
        Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
        → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

    2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
        Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
        → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

    3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
        Examples: "reserve these", "hold these for me", "check warehouse stock for X".
        → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

    Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
    "I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

    ## Available Agents

    - product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
    - shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
    - warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
    """

    template = Template(prompt_template)
    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort = "medium",
        use_responses_api=True,
    )

    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages #this lets the model reference the previous messages (the history)
        ]
    )

    final_answer = False
    answer = ""
    #plan = []
    next_agent = ""

    def sanitize_response(response):
        for tool_call in response.tool_calls:
            if tool_call.get('name') == "FinalAgentResponse":
                answer = tool_call.get('args').get('answer')
        
        return AIMessage(content=answer)    

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get('name') == 'Plan':
            #plan = response.tool_calls[0].get('args').get('plan')
            next_agent = response.tool_calls[0].get('args').get('next_agent')
            response = None
        else:   
            for tool_call in response.tool_calls:
                if tool_call.get('name') == 'FinalAgentResponse':
                    final_answer = True
                    answer = tool_call.get('args').get('answer')
                    response = sanitize_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "final_answer": final_answer,
            "iteration": state.coordinator_agent.iteration + 1,
            #"plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [15]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x['input']['messages'],
            coordinator_agent=CoordinatorProperties(
                iteration=0,
                plan=[],
                next_agent="",
                final_answer=False
            ),
            answer=""
        )
    ),
    data='coordinator-dataset-evaluation',
    evaluators=[evaluate_coordinator_delegation_for_ls],
    experiment_prefix='coordinator-delegation-v3',
    num_repetitions=5 #each example will be run 5 times and the results will be averaged
)

View the evaluation results for experiment: 'coordinator-delegation-v3-eac43c86' at:
https://smith.langchain.com/o/a215ca14-ac83-59e9-9b9e-fa7ba18e0446/datasets/268622a0-304b-495a-b347-361d21088c56/compare?selectedSessions=33f37139-d770-45c2-9a38-71eec719bd16




15it [00:59,  3.96s/it]


#### Option 4: Identifying the agent final answer in the response

In [16]:
@traceable(
    name= "coordinator_agent",
    run_type= "llm",
    metadata= {
        "ls_provider": "openai",
        "ls_model": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    prompt_template = """
    You are a Coordinator Agent as part of a shopping assistant.

    ## Instructions

    - Your role is to delegate work to worker agents in order to solve user queries.
    - You should output the next agent to invoke.
    - Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool or output the final answer via the `FinalAgentResponse` tool.
    - Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
    - Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
    - Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
    - Do not delegate work to the same agent twice in a row.

    ## Routing rules

    Match the user's latest message to one of these patterns:

    1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
        Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
        → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

    2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
        Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
        → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

    3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
        Examples: "reserve these", "hold these for me", "check warehouse stock for X".
        → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

    Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
    "I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

    ## Available Agents

    - product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
    - shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
    - warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
    """

    template = Template(prompt_template)
    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort = "medium",
        use_responses_api=True,
    )

    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages #this lets the model reference the previous messages (the history)
        ]
    )

    final_answer = False
    answer = ""
    #plan = []
    next_agent = ""

    def sanitize_response(response):
        for tool_call in response.tool_calls:
            if tool_call.get('name') == "FinalAgentResponse":
                answer = tool_call.get('args').get('answer')
        
        return AIMessage(content=answer)    

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get('name') == 'Plan':
            #plan = response.tool_calls[0].get('args').get('plan')
            next_agent = response.tool_calls[0].get('args').get('next_agent')
            response = None
        else:   
            for tool_call in response.tool_calls:
                if tool_call.get('name') == 'FinalAgentResponse':
                    final_answer = True
                    answer = tool_call.get('args').get('answer')
                    response = sanitize_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "final_answer": final_answer,
            "iteration": state.coordinator_agent.iteration + 1,
            #"plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [17]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x['input']['messages'],
            coordinator_agent=CoordinatorProperties(
                iteration=0,
                plan=[],
                next_agent="",
                final_answer=False
            ),
            answer=""
        )
    ),
    data='coordinator-dataset-evaluation-2',
    evaluators=[evaluate_coordinator_delegation_for_ls],
    experiment_prefix='coordinator-delegation-v1',
    num_repetitions=5 #each example will be run 5 times and the results will be averaged
)

View the evaluation results for experiment: 'coordinator-delegation-v1-51302a01' at:
https://smith.langchain.com/o/a215ca14-ac83-59e9-9b9e-fa7ba18e0446/datasets/34d56d06-7598-4453-bde0-5c8fab630bee/compare?selectedSessions=236bf0bb-2de1-429b-a510-cad9a1db36f7




15it [01:03,  4.23s/it]


### Option 5: Adding the coordinator agent decision before calling the next agent

In [18]:
@traceable(
    name= "coordinator_agent",
    run_type= "llm",
    metadata= {
        "ls_provider": "openai",
        "ls_model": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    prompt_template = """
    You are a Coordinator Agent as part of a shopping assistant.

    ## Instructions

    - Your role is to delegate work to worker agents in order to solve user queries.
    - You should output the next agent to invoke.
    - Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool or output the final answer via the `FinalAgentResponse` tool.
    - Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
    - Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
    - Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
    - Do not delegate work to the same agent twice in a row.

    ## Routing rules

    Match the user's latest message to one of these patterns:

    1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
        Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
        → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

    2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
        Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
        → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

    3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
        Examples: "reserve these", "hold these for me", "check warehouse stock for X".
        → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

    Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
    "I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

    ## Available Agents

    - product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
    - shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
    - warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
    """

    template = Template(prompt_template)
    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort = "medium",
        use_responses_api=True,
    )

    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages #this lets the model reference the previous messages (the history)
        ]
    )

    final_answer = False
    answer = ""
    #plan = []
    next_agent = ""

    def sanitize_response(response):
        for tool_call in response.tool_calls:
            if tool_call.get('name') == "FinalAgentResponse":
                answer = tool_call.get('args').get('answer')
        
        return AIMessage(content=answer)    

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get('name') == 'Plan':
            #plan = response.tool_calls[0].get('args').get('plan')
            next_agent = response.tool_calls[0].get('args').get('next_agent')
            response = None
        else:   
            for tool_call in response.tool_calls:
                if tool_call.get('name') == 'FinalAgentResponse':
                    final_answer = True
                    answer = tool_call.get('args').get('answer')
                    response = sanitize_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "final_answer": final_answer,
            "iteration": state.coordinator_agent.iteration + 1,
            #"plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [19]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x['input']['messages'],
            coordinator_agent=CoordinatorProperties(
                iteration=0,
                plan=[],
                next_agent="",
                final_answer=False
            ),
            answer=""
        )
    ),
    data='coordinator-dataset-evaluation-2',
    evaluators=[evaluate_coordinator_delegation_for_ls],
    experiment_prefix='coordinator-delegation-v2',
    num_repetitions=5 #each example will be run 5 times and the results will be averaged
)

View the evaluation results for experiment: 'coordinator-delegation-v2-171292fd' at:
https://smith.langchain.com/o/a215ca14-ac83-59e9-9b9e-fa7ba18e0446/datasets/34d56d06-7598-4453-bde0-5c8fab630bee/compare?selectedSessions=e87a5316-c991-45ba-a986-c60ef88fef86




15it [00:56,  3.75s/it]
